# Zero-Shot Classifier Evaluation
You-Are-Bot v2 Competition

In [ ]:
import json, csv
from pathlib import Path
import torch
from tqdm import tqdm
from transformers import pipeline
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
DATA = Path("data")
MODEL_NAME = "typeform/distilbert-base-uncased-mnli"
CANDIDATE_LABELS = ["bot", "human"]
HYPOTHESIS_TEMPLATE = "This message was written by a {}."
THRESHOLD = 0.5

## 1. Load Dataset

In [ ]:
print("Loading dataset...")
with open(DATA / "train.json") as f:
    dialogs = json.load(f)

rows = []
with open(DATA / "ytrain.csv") as f:
    for r in csv.DictReader(f):
        rows.append(r)

bot_count = sum(1 for r in rows if r["is_bot"] == "1")
human_count = sum(1 for r in rows if r["is_bot"] == "0")
print(f"  {len(dialogs)} dialogs, {len(rows)} labeled messages")
print(f"  Bot: {bot_count}, Human: {human_count}")

Loading dataset...
  1516 dialogs, 3032 labeled messages
  Bot: 1237, Human: 1795


In [ ]:
samples = []
for r in rows:
    did = r["dialog_id"]
    idx = int(r["participant_index"])
    label = int(r["is_bot"])
    if did in dialogs and idx < len(dialogs[did]):
        samples.append((dialogs[did][idx]["text"], label))

print(f"Matched {len(samples)} labeled texts")

Matched 3032 labeled texts


## 2. Load Zero-Shot Model

In [ ]:
print(f"Loading {MODEL_NAME}...")
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    hypothesis_template=HYPOTHESIS_TEMPLATE,
    device=-1,
)
print("Model loaded.")

Loading typeform/distilbert-base-uncased-mnli...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded.


## 3. Run Predictions

In [ ]:
y_true, y_pred = [], []
for text, label in tqdm(samples):
    result = classifier(text, candidate_labels=CANDIDATE_LABELS)
    bot_prob = float(result["scores"][result["labels"].index("bot")])
    y_pred.append(1 if bot_prob > THRESHOLD else 0)
    y_true.append(label)

  4%|▍         | 130/3032 [00:26<09:40,  5.00it/s]


KeyboardInterrupt: 

## 4. Results

In [ ]:
print(classification_report(y_true, y_pred, target_names=["human", "bot"]))
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

## 5. Generate Submission (Test Set)

In [ ]:
print("Loading test.json...")
with open(DATA / "test.json") as f:
    test_dialogs = json.load(f)

rows_test = []
with open(DATA / "ytest.csv") as f:
    for r in csv.DictReader(f):
        rows_test.append(r)

test_samples = []
for r in rows_test:
    did = r["dialog_id"]
    idx = int(r["participant_index"])
    if did in test_dialogs and idx < len(test_dialogs[did]):
        test_samples.append((r["ID"], test_dialogs[did][idx]["text"]))

test_preds = []
for id_, text in tqdm(test_samples):
    result = classifier(text, candidate_labels=CANDIDATE_LABELS)
    prob = float(result["scores"][result["labels"].index("bot")])
    test_preds.append({"ID": id_, "is_bot": prob})

out = DATA / "submission.csv"
with open(out, "w") as f:
    w = csv.DictWriter(f, fieldnames=["ID", "is_bot"])
    w.writeheader()
    w.writerows(test_preds)
print(f"Saved {out} ({len(test_preds)} rows)")
